# Multi-Wavelength Galaxy Classifier
Trains a ResNet-18 model on composite (IR, VIS, UV) galaxy images.

Classes: **Spiral**, **Elliptical**, **Anomaly**

Checkpoints are mounted after every epoch to Google Drive for failsafe progress.

In [ ]:
# 1. Mount Google Drive and verify GPU
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

In [ ]:
# 2. Setup paths
import os

DRIVE_DIR = "/content/drive/MyDrive/GalaxyProject"
CKPT_DIR = os.path.join(DRIVE_DIR, "checkpoints")
DATA_DIR = "/content/galaxy_data"

os.makedirs(CKPT_DIR, exist_ok=True)
print("Checkpoints directory:", CKPT_DIR)

In [ ]:
# 3. Unzip dataset
drive_zip = os.path.join(DRIVE_DIR, "galaxy_dataset.zip")
local_zip = "/content/galaxy_dataset.zip"

if os.path.exists(drive_zip):
    print(f"Unzipping from Drive: {drive_zip}")
    !unzip -q "{drive_zip}" -d "{DATA_DIR}"
elif os.path.exists(local_zip):
    print(f"Unzipping from local storage: {local_zip}")
    !unzip -q "{local_zip}" -d "{DATA_DIR}"
else:
    print("Please upload galaxy_dataset.zip to Google Drive (GalaxyProject/) or drag it to Colab.")

print("Ready!")

In [ ]:
# 4. Training Pipeline
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Random 90-degree rotations for astronomical images
class Rotate90:
    def __init__(self):
        self.angles = [0, 90, 180, 270]
    def __call__(self, img):
        return transforms.functional.rotate(img, random.choice(self.angles))

def get_transforms(img_size=224):
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        Rotate90(),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf

def get_model(num_classes=3):
    model = models.resnet18(weights="DEFAULT")
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

def train(data_dir, ckpt_dir, epochs=25, batch_size=8, lr=3e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")

    train_tf, val_tf = get_transforms()
    train_set = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_tf)
    val_set = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=val_tf)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2)
    classes = train_set.classes
    print(f"Classes: {classes}")

    # Inverse frequency class weighting
    targets = [s[1] for s in train_set.samples]
    _, counts = np.unique(targets, return_counts=True)
    weights = len(targets) / (len(counts) * counts.astype(np.float32))
    class_weights = torch.tensor(weights, dtype=torch.float, device=device)

    model = get_model(num_classes=len(classes)).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    last_ckpt = os.path.join(ckpt_dir, "last_checkpoint.pth")
    best_ckpt = os.path.join(ckpt_dir, "best_model.pth")
    start_epoch = 1
    best_acc = 0.0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    # Resume from checkpoint if found in Drive
    if os.path.exists(last_ckpt):
        print(f"Resuming from: {last_ckpt}")
        ckpt = torch.load(last_ckpt, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        best_acc = ckpt.get("best_acc", 0.0)
        history = ckpt.get("history", history)
        print(f"Resumed at Epoch {start_epoch} (Best val acc: {best_acc*100:.2f}%)")

    for epoch in range(start_epoch, epochs + 1):
        # Train
        model.train()
        t_loss, t_corr, t_tot = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            t_loss += loss.item() * x.size(0)
            t_corr += (out.argmax(1) == y).sum().item()
            t_tot += y.size(0)

        # Validate
        model.eval()
        v_loss, v_corr, v_tot = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                loss = criterion(out, y)
                v_loss += loss.item() * x.size(0)
                v_corr += (out.argmax(1) == y).sum().item()
                v_tot += y.size(0)

        train_loss, train_acc = t_loss / t_tot, t_corr / t_tot
        val_loss, val_acc = v_loss / v_tot, v_corr / v_tot
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch [{epoch:02d}/{epochs:02d}] Train Loss: {train_loss:.4f} Acc: {train_acc*100:.1f}% | Val Loss: {val_loss:.4f} Acc: {val_acc*100:.1f}%")

        is_best = val_acc > best_acc
        if is_best:
            best_acc = val_acc

        # Save checkpoint after every epoch
        state = {
            "epoch": epoch,
            "arch": "resnet18",
            "class_names": classes,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_acc": best_acc,
            "history": history
        }
        torch.save(state, last_ckpt)
        if is_best:
            torch.save(state, best_ckpt)
            print(f"  -> Best model saved to Drive! ({val_acc*100:.1f}%)")

    print(f"Done! Best validation accuracy: {best_acc*100:.1f}%")
    return history, best_acc, classes

In [ ]:
# 5. Training
history, best_acc, classes = train(DATA_DIR, CKPT_DIR, epochs=25, batch_size=8, lr=3e-4)

In [ ]:
# 6. Plot Loss and Accuracy
epochs_range = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], 'b-', label="Train")
plt.plot(epochs_range, history["val_loss"], 'r-', label="Val")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["train_acc"], 'b-', label="Train")
plt.plot(epochs_range, history["val_acc"], 'r-', label="Val")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 7. Test Prediction on a Galaxy
from PIL import Image
import torch.nn.functional as F

def test_galaxy(image_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(os.path.join(CKPT_DIR, "best_model.pth"), map_location=device)
    model = get_model(num_classes=3).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    _, val_tf = get_transforms()
    img = Image.open(image_path).convert("RGB")
    tensor = val_tf(img).unsqueeze(0).to(device)

    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1).cpu().numpy()[0]

    pred = classes[probs.argmax()]

    plt.figure(figsize=(7, 3))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Pred: {pred.upper()} ({probs.max()*100:.1f}%)")

    plt.subplot(1, 2, 2)
    plt.barh([c.capitalize() for c in classes], probs * 100, color=["#2b5c8f", "#d97724", "#bd3333"])
    plt.xlim(0, 100)
    plt.xlabel("Confidence (%)")
    plt.tight_layout()
    plt.show()

# Test on a sample test galaxy:
test_galaxy(f"{DATA_DIR}/test/spiral/M74.png")